# Destek 04 — Sağkalım (Görsel 5: Hayatta kalma devrimi)

**Amaç:** "1000 bebekten kaçı 1, 5, 40 ve 60 yaşını görüyor?" sorusunun verisini üretmek. Kontrol listesi #6'yı hesaplamak.

**Girdi:** `data/clean/countries_clean.csv` (2101 dışarıda). Sütunlar: `IMR`, `Q5`, `Q0040`, `Q0060` (ve cinsiyete göre `Q0040Male/Female`, `Q0060Male/Female`). Bunlar **1000 canlı doğum başına** ölüm olasılığıdır, bölme ya da ölçekleme gerekmez.

**Hesap:** `survivors_per_1000 = 1000 − ölüm_oranı`. 0 yaş = 1000; 1 yaş = 1000 − IMR; 5 yaş = 1000 − Q5; 40 yaş = 1000 − Q0040; 60 yaş = 1000 − Q0060.

**Çıktı:** `data/out/g05_sagkalim.csv` (country, year, sex, age, survivors_per_1000).

**Önemli (sunumda bir satırla söylenecek):** Bunlar *dönem olasılıklarıdır*: "o yılın ölüm koşulları ömür boyu sürseydi ne olurdu?" sorusunun cevabı. Gerçek bir kuşağın (örneğin 1950'de doğanların) yaşadığı değil.

In [1]:
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").exists())
OUT = ROOT / "data/out"; OUT.mkdir(parents=True, exist_ok=True)
raw = pd.read_csv(ROOT / "data/clean/countries_clean.csv")
df = raw[raw.Time <= 2100].copy()
print("Ülke:", df.LocID.nunique(), "| yıl:", df.Time.min(), "-", df.Time.max())

# Kapsam
ODAK = "Türkiye"; ODAK_YILLAR = [1950, 1990, 2023]
KARSILASTIRMA = ["Japan", "Nigeria", "Slovenia", "Portugal", "Italy", "Germany", "Greece", "Republic of Korea"]   # yol haritası: Japonya, Nijerya; liderin önerisi: 7 ülke
KARS_YIL = 2023
eksik = [x for x in [ODAK, *KARSILASTIRMA] if x not in set(df.Location)]
assert not eksik, eksik

Ülke: 237 | yıl: 1950 - 2100


## 1. Girdi sağlamlığı

Sağkalım eğrisi azalan olmalı: `IMR ≤ Q5 ≤ Q0040 ≤ Q0060` (yaş büyüdükçe ölmüş olanların toplamı artar). Tüm ülke-yıllarda sınanır.

In [2]:
kontrol = pd.DataFrame([
 ("IMR > Q5 olan satır",         int((df.IMR > df.Q5 + 1e-9).sum()), 0),
 ("Q5 > Q0040 olan satır",       int((df.Q5 > df.Q0040 + 1e-9).sum()), 0),
 ("Q0040 > Q0060 olan satır",    int((df.Q0040 > df.Q0060 + 1e-9).sum()), 0),
 ("Q0060 > 1000 olan satır",     int((df.Q0060 > 1000).sum()), 0),
 ("Cinsiyete göre Q eksik",      int(df[["Q0040Male", "Q0040Female", "Q0060Male", "Q0060Female"]].isna().sum().sum()), 0),
 ("IMR/Q5/Q0040/Q0060 eksik",    int(df[["IMR", "Q5", "Q0040", "Q0060"]].isna().sum().sum()), 0),
], columns=["kontrol", "bulunan", "beklenen"])
kontrol["tamam"] = kontrol.bulunan == kontrol.beklenen
display(kontrol)
print("Birim kontrolü (Türkiye 1950): IMR =", round(df[(df.Location == ODAK) & (df.Time == 1950)].IMR.iloc[0], 1), "→ 1000 doğumda yaklaşık 193 bebek 1 yaşına gelmeden ölüyor.")

,kontrol,bulunan,beklenen,tamam
0,IMR > Q5 olan satır,0,0,True
1,Q5 > Q0040 olan satır,0,0,True
2,Q0040 > Q0060 olan satır,0,0,True
3,Q0060 > 1000 olan satır,0,0,True
4,Cinsiyete göre Q eksik,0,0,True
5,IMR/Q5/Q0040/Q0060 eksik,0,0,True


Birim kontrolü (Türkiye 1950): IMR = 193.4 → 1000 doğumda yaklaşık 193 bebek 1 yaşına gelmeden ölüyor.


## 2. Hesap

Cinsiyetlere göre yalnızca 40 ve 60 yaş için veri var (`Q0040Male/Female`, `Q0060Male/Female`). 1 ve 5 yaşta cinsiyete göre ayrım **yok**; `sex = both` satırlarında bulunur. `sex = male/female` satırlarında yaşlar 0, 40, 60'tır.

In [3]:
KOL = {"both":   {1: "IMR", 5: "Q5", 40: "Q0040", 60: "Q0060"},
       "male":   {40: "Q0040Male", 60: "Q0060Male"},
       "female": {40: "Q0040Female", 60: "Q0060Female"}}

secim = pd.concat([df[(df.Location == ODAK) & df.Time.isin(ODAK_YILLAR)],
                   df[df.Location.isin(KARSILASTIRMA) & (df.Time == KARS_YIL)]])
satirlar = []
for _, r in secim.iterrows():
    for sex, yaslar in KOL.items():
        satirlar.append((r.Location, int(r.Time), sex, 0, 1000.0))
        for yas, kol in yaslar.items():
            satirlar.append((r.Location, int(r.Time), sex, yas, round(1000 - r[kol], 1)))
g05 = pd.DataFrame(satirlar, columns=["country", "year", "sex", "age", "survivors_per_1000"])
g05["country"] = g05.country.replace({"Türkiye": "Türkiye"})
g05 = g05.sort_values(["country", "year", "sex", "age"]).reset_index(drop=True)
print(g05.shape); display(g05[(g05.country == ODAK) & (g05.sex == "both")].pivot(index="year", columns="age", values="survivors_per_1000"))

(121, 5)


age,0,1,5,40,60
year,,,,,
1950,1000.0,806.6,689.8,623.4,513.9
1990,1000.0,941.3,926.2,890.9,778.7
2023,1000.0,990.8,986.9,963.5,895.0


## 3. Kontrol sayıları (kontrol listesi #6)

In [4]:
tr = g05[(g05.country == ODAK) & (g05.sex == "both")].pivot(index="year", columns="age", values="survivors_per_1000")
beklenen = {1950: (690, 623, 514), 1990: (926, 891, 779), 2023: (987, 964, 895)}
rows = []
for y, (b5, b40, b60) in beklenen.items():
    h = [int(round(tr.loc[y, a])) for a in (5, 40, 60)]
    rows.append((y, *h, b5, b40, b60))
kt = pd.DataFrame(rows, columns=["yıl", "5_yaş", "40_yaş", "60_yaş", "beklenen_5", "beklenen_40", "beklenen_60"])
kt["tamam"] = (kt["5_yaş"] == kt.beklenen_5) & (kt["40_yaş"] == kt.beklenen_40) & (kt["60_yaş"] == kt.beklenen_60)
display(kt)
print("Yuvarlama kuralı: CSV'de 1 ondalık; slayt ve metinde en yakın tam sayı (514,0 → 514). Tüm slaytlarda aynı kural.")

,yıl,5_yaş,40_yaş,60_yaş,beklenen_5,beklenen_40,beklenen_60,tamam
0,1950,690,623,514,690,623,514,True
1,1990,926,891,779,926,891,779,True
2,2023,987,964,895,987,964,895,True


Yuvarlama kuralı: CSV'de 1 ondalık; slayt ve metinde en yakın tam sayı (514,0 → 514). Tüm slaytlarda aynı kural.


## 4. Karşılaştırma ve cinsiyet farkı

In [5]:
k = g05[(g05.year == KARS_YIL) & (g05.sex == "both") & (g05.age == 60)].set_index("country").survivors_per_1000.sort_values()
display(k.round(0).astype(int).to_frame("60_yaşa_ulaşan_(1000_doğumda)"))
cin = g05[(g05.country == ODAK) & (g05.age == 60) & g05.sex.isin(["male", "female"])].pivot(index="year", columns="sex", values="survivors_per_1000")
cin["fark_(kadın−erkek)"] = (cin.female - cin.male).round(1)
display(cin.round(1))

,60_yaşa_ulaşan_(1000_doğumda)
country,
Nigeria,553
Türkiye,895
Portugal,930
Germany,933
Slovenia,938
Greece,940
Japan,949
Italy,951
Republic of Korea,964


sex,female,male,fark_(kadın−erkek)
year,,,
1950,567.3,461.8,105.5
1990,831.4,728.1,103.3
2023,918.1,872.4,45.7


**Okuma:** Türkiye'de 1000 doğumdan 60 yaşına ulaşan sayısı 1950'de 514, 1990'da 779, 2023'te 895. 2023'te Türkiye, Nijerya'nın çok üstünde, karşılaştırma ülkelerinin ise altında. Kadınlar 60 yaşa ulaşmada erkeklerin önünde ve fark zamanla daralıyor.

## 5. Çıktı

In [6]:
assert g05.survivors_per_1000.between(0, 1000).all() and g05.isna().sum().sum() == 0
assert not g05.duplicated(["country", "year", "sex", "age"]).any()
# eğri azalan mı (her ülke-yıl-cinsiyet için yaş arttıkça düşmeli)
assert (g05.sort_values("age").groupby(["country", "year", "sex"]).survivors_per_1000.apply(lambda s: s.is_monotonic_decreasing)).all()
g05.to_csv(OUT / "g05_sagkalim.csv", index=False, encoding="utf-8")
print("Yazıldı:", OUT / "g05_sagkalim.csv", g05.shape)
display(g05.groupby(["sex"]).size().to_frame("satır"))

Yazıldı: /home/claude/repo/data/out/g05_sagkalim.csv (121, 5)


,satır
sex,
both,55
female,33
male,33


**Görselleştirmeciye notlar**

- Noktalar yalnızca 0, 1, 5, 40 ve 60 yaşlarında var; **40 ile 60 arasında veri yok**. Noktaları düz çizgiyle birleştirmeyin (basamaklı eğri ya da yalnızca işaretçiler).
- Cinsiyete göre satırlarda (`male`, `female`) yalnızca 0, 40, 60 yaşları var.
- Grafikte altyazı: "Dönem olasılığı: o yılın ölüm koşulları ömür boyu sürseydi."
- `country` değeri Türkiye için `Türkiye`; diğer ülkeler BM'nin İngilizce adlarıyla (`Japan`, `Republic of Korea` vb.). Sunumda Türkçe adlandırma görselleştirmecide.

In [7]:
t = tr
print(f"BULGU (g05):  Türkiye'de 1000 doğumdan 60 yaşına ulaşan sayısı {int(round(t.loc[1950,60]))} (1950) → {int(round(t.loc[1990,60]))} (1990) → {int(round(t.loc[2023,60]))} (2023).")
print(f"NOT:          1950'de 5 yaşını göremeyen {int(round(1000 - t.loc[1950,5]))} çocuk vardı; 2023'te {int(round(1000 - t.loc[2023,5]))}. (Dönem olasılığı, gerçek bir kuşak değil.)")
ok = "Hesaplandı, kontrol bekliyor"
display(pd.DataFrame([(6, "1000 doğumdan 60'a ulaşan: 514 → 779 → 895", "Destek", "Lider", ok if kt.tamam.all() else "Fark var")],
                     columns=["#", "sayı", "hesaplayan", "kontrol eden", "durum"]))

BULGU (g05):  Türkiye'de 1000 doğumdan 60 yaşına ulaşan sayısı 514 (1950) → 779 (1990) → 895 (2023).
NOT:          1950'de 5 yaşını göremeyen 310 çocuk vardı; 2023'te 13. (Dönem olasılığı, gerçek bir kuşak değil.)


,#,sayı,hesaplayan,kontrol eden,durum
0,6,1000 doğumdan 60'a ulaşan: 514 → 779 → 895,Destek,Lider,"Hesaplandı, kontrol bekliyor"
